# MoCap Analysis — Transform to TFrame Coordinate System

**Workflow:**
- `multiple_rigid_body.py` is loaded once via `importlib`.
- It runs its `__main__` block which produces `rb_dfs` and `st_time`.
- This notebook uses those DataFrames directly — no CSV re-read.

**Pipeline:**
1. Imports
2. Load script → get `rb_dfs` and `st_time`
3. Verify DataFrames are ready
4. Inspect raw data
5. Define and run coordinate frame transform
6. Sanity checks
7. Print noark m5 position in tframe (metres)
8. Print table m1–m5 positions in tframe (metres)

## Cell 1 — Imports

In [ ]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R

print('Libraries loaded OK')

## Cell 2 — Load `multiple_rigid_body.py`

Walks up from the notebook directory to find the repo root,
then loads the script via `importlib`. The script's `__main__` block
runs automatically and populates `rb_dfs` and `st_time` into this
notebook's namespace.

In [ ]:
import sys, importlib.util
from pathlib import Path

# ── locate this notebook's directory ─────────────────────────────────────────
try:
    _nb_file = Path(globals()['__vsc_ipynb_file__']).resolve()
    _nb_dir  = _nb_file.parent
except KeyError:
    _nb_dir = Path.cwd()

# ── walk up to find repo root ─────────────────────────────────────────────────
repo_root = _nb_dir
for candidate in [_nb_dir, *_nb_dir.parents]:
    if (candidate / 'vaideesh').exists() or (candidate / '.venv').exists():
        repo_root = candidate
        break

print(f'Repo root : {repo_root}')

# ── add script directory to sys.path so local imports inside the module work ──
_script_path = repo_root / 'vaideesh' / 'Analysis' / 'multiple_rigid_body.py'
assert _script_path.exists(), f'Script not found: {_script_path}'

_script_dir = str(_script_path.parent)
if _script_dir not in sys.path:
    sys.path.insert(0, _script_dir)

# ── load the module ───────────────────────────────────────────────────────────
spec = importlib.util.spec_from_file_location('multiple_rigid_body', _script_path)
mrb  = importlib.util.module_from_spec(spec)
sys.modules['multiple_rigid_body'] = mrb
spec.loader.exec_module(mrb)

# ── call the read function directly (since __main__ block doesn't run) ────────
FILE = "E:/Ragav/MS Bio Engineering/NOARK_backbone/mocap_data/table_frame.csv"

rb_dfs, st_time = mrb.read_3_rigid_body_csv(
    FILE,
    rb_names=["tframe", "noark", "table"],
)
rb_dfs = mrb.add_datetime_col_3rb(rb_dfs, st_time)
rb_dfs["tframe"], rb_dfs["noark"], rb_dfs["table"] = mrb.trunkate_3_dfs(
    rb_dfs["tframe"], rb_dfs["noark"], rb_dfs["table"], display_print=True
)

# ── bring helper functions into notebook namespace directly from mrb ───────────
add_datetime_col_3rb   = mrb.add_datetime_col_3rb
trunkate_3_dfs         = mrb.trunkate_3_dfs
get_rb_pos_cols        = mrb.get_rb_pos_cols
get_rb_rot_cols        = mrb.get_rb_rot_cols
get_rb_marker_name_3rb = mrb.get_rb_marker_name_3rb

print(f'Script loaded      : {_script_path}')
print(f'Capture start time : {st_time}')
print('rb_dfs keys        :', list(rb_dfs.keys()))

## Cell 3 — Verify DataFrames

The `__main__` block in `multiple_rigid_body.py` already calls
`add_datetime_col_3rb()` and `trunkate_3_dfs()`, so the DataFrames
are ready to use as-is. This cell just confirms their shapes and
that a `time` column is present.

In [ ]:
print('DataFrames from multiple_rigid_body.py:')
print(f'{"Body":<10}  {"Shape":<15}  {"Has time col"}')
print('-' * 40)
for name, df in rb_dfs.items():
    has_time = 'time' in df.columns
    print(f'{name:<10}  {str(df.shape):<15}  {has_time}')

print('\nNaN rows per body:')
for name, df in rb_dfs.items():
    nan_rows = df.isna().any(axis=1).sum()
    print(f'  {name}: {nan_rows} NaN rows out of {len(df)}')

## Cell 4 — Inspect raw DataFrames

In [ ]:
print('── tframe ──')
display(rb_dfs['tframe'].head(3))

In [ ]:
print('── noark ──')
display(rb_dfs['noark'].head(3))

In [ ]:
print('── table ──')
display(rb_dfs['table'].head(3))

## Cell 5 — Define `transform_to_tframe()`

**Math per frame:**

| Step | Formula | Notes |
|---|---|---|
| Translate | `Δp = p_world − p_tframe` | remove tframe origin offset |
| Rotate    | `p_rel = R_tframe⁻¹ · Δp` | align axes to tframe |
| Orientation | `q_rel = q_tframe⁻¹ ⊗ q_target` | relative rotation |
| Euler | `scipy.Rotation.as_euler('xyz', degrees=True)` | human-readable angles |

**Units:** Motive exports positions in **metres** — no conversion needed.  
**Column suffix `_m`** makes the unit explicit in every position column.

In [ ]:
def transform_to_tframe(rb_dfs):
    tframe_df = rb_dfs['tframe'].copy()
    noark_df  = rb_dfs['noark'].copy()
    table_df  = rb_dfs['table'].copy()

    quat_cols_tf    = ['tframe_rot_x', 'tframe_rot_y', 'tframe_rot_z', 'tframe_rot_w']
    quat_cols_noark = ['noark_rot_x',  'noark_rot_y',  'noark_rot_z',  'noark_rot_w']
    quat_cols_table = ['table_rot_x',  'table_rot_y',  'table_rot_z',  'table_rot_w']

    # ── drop rows where ANY of the 3 bodies has a bad quaternion ─────────────
    def bad_quat_mask(df, cols):
        q = df[cols].values.astype(float)
        return (
            np.any(np.isnan(q), axis=1) |
            (np.linalg.norm(q, axis=1) < 1e-6)
        )

    bad_mask = (
        bad_quat_mask(tframe_df, quat_cols_tf)    |
        bad_quat_mask(noark_df,  quat_cols_noark) |
        bad_quat_mask(table_df,  quat_cols_table)
    )

    n_bad = bad_mask.sum()
    if n_bad > 0:
        print(f"[transform_to_tframe] Dropping {n_bad} / {len(tframe_df)} rows "
              f"with invalid quaternions across any body.")
        good_idx  = tframe_df.index[~bad_mask]
        tframe_df = tframe_df.loc[good_idx].reset_index(drop=True)
        noark_df  = noark_df.loc[good_idx].reset_index(drop=True)
        table_df  = table_df.loc[good_idx].reset_index(drop=True)

    # ── now safe to build rotations ───────────────────────────────────────────
    q_tf = tframe_df[quat_cols_tf].values.astype(float)
    p_tf = tframe_df[['tframe_pos_x', 'tframe_pos_y', 'tframe_pos_z']].values.astype(float)

    rot_tf_inv = R.from_quat(q_tf).inv()

    def pos_to_tframe(p_world):
        return rot_tf_inv.apply(p_world - p_tf)

    def rot_to_tframe(q_world_xyzw):
        return rot_tf_inv * R.from_quat(q_world_xyzw)

    # ── noark ─────────────────────────────────────────────────────────────────
    noark_result = pd.DataFrame({
        'frame'  : noark_df['frame'].values,
        'seconds': noark_df['seconds'].values,
        'time'   : noark_df['time'].values,
    })

    p_m5    = noark_df[['noark_marker_m5_x',
                         'noark_marker_m5_y',
                         'noark_marker_m5_z']].values.astype(float)
    p_m5_tf = pos_to_tframe(p_m5)
    noark_result['noark_m5_tf_x_m'] = p_m5_tf[:, 0]
    noark_result['noark_m5_tf_y_m'] = p_m5_tf[:, 1]
    noark_result['noark_m5_tf_z_m'] = p_m5_tf[:, 2]

    rot_noark_tf   = rot_to_tframe(noark_df[quat_cols_noark].values.astype(float))
    q_noark_tf     = rot_noark_tf.as_quat()
    euler_noark_tf = rot_noark_tf.as_euler('xyz', degrees=True)

    noark_result['noark_quat_tf_x']    = q_noark_tf[:, 0]
    noark_result['noark_quat_tf_y']    = q_noark_tf[:, 1]
    noark_result['noark_quat_tf_z']    = q_noark_tf[:, 2]
    noark_result['noark_quat_tf_w']    = q_noark_tf[:, 3]
    noark_result['noark_roll_tf_deg']  = euler_noark_tf[:, 0]
    noark_result['noark_pitch_tf_deg'] = euler_noark_tf[:, 1]
    noark_result['noark_yaw_tf_deg']   = euler_noark_tf[:, 2]

    # ── table ─────────────────────────────────────────────────────────────────
    table_result = pd.DataFrame({
        'frame'  : table_df['frame'].values,
        'seconds': table_df['seconds'].values,
        'time'   : table_df['time'].values,
    })

    for i in range(1, 6):
        p_world     = table_df[[f'table_marker_m{i}_x',
                                 f'table_marker_m{i}_y',
                                 f'table_marker_m{i}_z']].values.astype(float)
        p_tf_coords = pos_to_tframe(p_world)
        table_result[f'table_m{i}_tf_x_m'] = p_tf_coords[:, 0]
        table_result[f'table_m{i}_tf_y_m'] = p_tf_coords[:, 1]
        table_result[f'table_m{i}_tf_z_m'] = p_tf_coords[:, 2]

    rot_table_tf   = rot_to_tframe(table_df[quat_cols_table].values.astype(float))
    q_table_tf     = rot_table_tf.as_quat()
    euler_table_tf = rot_table_tf.as_euler('xyz', degrees=True)

    table_result['table_quat_tf_x']    = q_table_tf[:, 0]
    table_result['table_quat_tf_y']    = q_table_tf[:, 1]
    table_result['table_quat_tf_z']    = q_table_tf[:, 2]
    table_result['table_quat_tf_w']    = q_table_tf[:, 3]
    table_result['table_roll_tf_deg']  = euler_table_tf[:, 0]
    table_result['table_pitch_tf_deg'] = euler_table_tf[:, 1]
    table_result['table_yaw_tf_deg']   = euler_table_tf[:, 2]

    return {
        'noark_in_tframe': noark_result.reset_index(drop=True),
        'table_in_tframe': table_result.reset_index(drop=True),
    }

## Cell 6 — Run the transformation

In [ ]:
tframe_dfs = transform_to_tframe(rb_dfs)

print('noark_in_tframe shape :', tframe_dfs['noark_in_tframe'].shape)
print('table_in_tframe shape :', tframe_dfs['table_in_tframe'].shape)
print('\nnoark_in_tframe columns:')
print(tframe_dfs['noark_in_tframe'].columns.tolist())
print('\ntable_in_tframe columns:')
print(tframe_dfs['table_in_tframe'].columns.tolist())

## Cell 7 — Inspect transformed DataFrames

In [ ]:
print('── noark in tframe ──')
display(tframe_dfs['noark_in_tframe'].head())

In [ ]:
print('── table in tframe ──')
display(tframe_dfs['table_in_tframe'].head())

## Cell 8 — Sanity checks

In [ ]:
# Check 1: tframe position transformed into itself → should be (0, 0, 0) m
p_tf_raw = rb_dfs['tframe'][['tframe_pos_x','tframe_pos_y','tframe_pos_z']].values.astype(float)
q_tf_raw = rb_dfs['tframe'][['tframe_rot_x','tframe_rot_y','tframe_rot_z','tframe_rot_w']].values.astype(float)

# ── drop bad quaternion rows ──────────────────────────────────────────────────
bad_mask = (
    np.any(np.isnan(q_tf_raw), axis=1) |
    (np.linalg.norm(q_tf_raw, axis=1) < 1e-6)
)
q_tf = q_tf_raw[~bad_mask]
p_tf = p_tf_raw[~bad_mask]

print(f'Dropped {bad_mask.sum()} bad rows for sanity checks.')

# ── Check 1 ───────────────────────────────────────────────────────────────────
rot_tf_inv  = R.from_quat(q_tf).inv()
self_pos    = rot_tf_inv.apply(p_tf - p_tf)
max_pos_err = np.nanmax(np.abs(self_pos))
print(f'Check 1 — self-position = 0 m     : max error = {max_pos_err:.2e} m  →  {"PASS ✓" if max_pos_err < 1e-10 else "FAIL ✗"}')

# ── Check 2 ───────────────────────────────────────────────────────────────────
self_rot    = (rot_tf_inv * R.from_quat(q_tf)).as_quat()
identity    = np.array([0., 0., 0., 1.])
max_rot_err = np.nanmax(np.abs(self_rot - identity))
print(f'Check 2 — self-rotation = identity : max deviation = {max_rot_err:.2e}  →  {"PASS ✓" if max_rot_err < 1e-6 else "FAIL ✗"}')
print(f'          row 0 quaternion          = {self_rot[0].round(6)}')

In [ ]:
# Check 3: all output quaternion norms should be ~1.0
for label, xyzw_cols in [
    ('noark', ['noark_quat_tf_x','noark_quat_tf_y','noark_quat_tf_z','noark_quat_tf_w']),
    ('table', ['table_quat_tf_x','table_quat_tf_y','table_quat_tf_z','table_quat_tf_w']),
]:
    q     = tframe_dfs[f'{label}_in_tframe'][xyzw_cols].values.astype(float)
    norms = np.linalg.norm(q, axis=1)
    valid = ~np.isnan(norms)
    if valid.sum() == 0:
        print(f'Check 3 — {label} : no valid rows to check')
        continue
    max_dev = np.nanmax(np.abs(norms[valid] - 1.0))   # nanmax not max
    print(f'Check 3 — {label} quat norms ≈ 1.0 : max deviation = {max_dev:.2e}  →  {"PASS ✓" if max_dev < 1e-5 else "FAIL ✗"}')

## Cell 9 — noark m5 position in tframe (metres)

In [ ]:
# Cell 9 — noark m5
df_n = tframe_dfs['noark_in_tframe']
print('noark Marker 5 — position in tframe coordinate system (metres)')
print(df_n[['frame','seconds','noark_m5_tf_x_m','noark_m5_tf_y_m','noark_m5_tf_z_m']].to_string(index=False))
print('\nSummary statistics (metres):')
display(df_n[['noark_m5_tf_x_m','noark_m5_tf_y_m','noark_m5_tf_z_m']].describe().round(6))

## Cell 10 — Table marker m1–m5 positions in tframe (metres)

In [ ]:
# Cell 10 — table m1–m5
df_t = tframe_dfs['table_in_tframe']
for i in range(1, 6):
    cols = [f'table_m{i}_tf_x_m', f'table_m{i}_tf_y_m', f'table_m{i}_tf_z_m']
    print(f'\ntable Marker {i} — position in tframe coordinate system (metres)')
    print(df_t[['frame','seconds'] + cols].to_string(index=False))
    print(f'\nSummary statistics — Marker {i} (metres):')
    display(df_t[cols].describe().round(6))